# M3 · Projection and Subspaces — companion notebook

> **Play with this.** A *demonstration, not an assessment* — the module's real assessment is its problem set. This notebook runs the module's three pictures live: projection onto a line with its orthogonal residual, OLS revealed as projection via the hat matrix, and a semantic axis you build, use, and then deliberately break.

Companion to the **Projection and Subspaces** module of the Mathematical Foundations track at [llmsforsocialscience.net](https://llmsforsocialscience.net/).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(3)

## 1 · Projection onto a line, and the right angle

The one-line formula, then the check that defines it: the residual is orthogonal to the direction projected onto.

In [ ]:
def project(a, b):
    return (a @ b) / (b @ b) * b

a = np.array([3.0, 1.0])
b = np.array([2.0, 2.0])
p = project(a, b)
r = a - p

print(f"projection: {p}")
print(f"residual:   {r}")
print(f"residual · b = {r @ b:.10f}   <- orthogonal, to machine precision")

fig, ax = plt.subplots(figsize=(5, 5))
ax.axline((0, 0), b, color="lightgray", lw=1.5, label="span(b)")
ax.annotate("", xy=a, xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="tab:blue", lw=2))
ax.annotate("", xy=p, xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="tab:red", lw=2))
ax.plot([a[0], p[0]], [a[1], p[1]], "k--", lw=1, label="residual")
ax.text(*a, "  a"); ax.text(*p, "  proj")
ax.set_aspect("equal"); ax.set_xlim(-0.5, 3.5); ax.set_ylim(-0.5, 3.5); ax.legend()
plt.tight_layout(); plt.show()

## 2 · OLS is projection — the hat matrix, verified

Fit a regression two ways: with `lstsq` (what your software does) and by building the projection matrix $P = X(X^\top X)^{-1}X^\top$ and projecting $y$ directly. Same fitted values, to machine precision — because they are the same operation.

In [ ]:
n = 50
x = rng.uniform(0, 10, n)
y = 2.0 + 0.7 * x + rng.normal(0, 1.2, n)
X = np.column_stack([np.ones(n), x])          # intercept + one regressor

beta_lstsq, *_ = np.linalg.lstsq(X, y, rcond=None)
P = X @ np.linalg.inv(X.T @ X) @ X.T           # the hat matrix
y_hat_proj = P @ y                              # ... is a projection
y_hat_lstsq = X @ beta_lstsq

print(f"max |difference| between the two fitted-value routes: {np.abs(y_hat_proj - y_hat_lstsq).max():.2e}")

e = y - y_hat_proj
print(f"residual · intercept column = {e @ X[:, 0]:.2e}   (residuals sum to zero)")
print(f"residual · regressor column = {e @ X[:, 1]:.2e}   ('uncorrelated with predictors')")
print(f"P @ P equals P (max entry diff): {np.abs(P @ P - P).max():.2e}   (a shadow of a shadow is itself)")

Every line above is a module claim, checked numerically: fitted values are $Py$; the residual is orthogonal to every column (including the ones column — residuals sum to zero); and $P^2 = P$.

## 3 · Projection onto a subspace, coordinate by coordinate

With an orthonormal basis, subspace projection is additive: project onto each direction with the one-line formula and sum. Verify against the hat-matrix route on the same subspace.

In [ ]:
# a 2-dimensional subspace of R^5, spanned by two random directions
A = rng.standard_normal((5, 2))
Q, _ = np.linalg.qr(A)               # Gram–Schmidt, industrial strength: orthonormal basis
v = rng.standard_normal(5)

proj_additive = (v @ Q[:, 0]) * Q[:, 0] + (v @ Q[:, 1]) * Q[:, 1]
proj_hat = A @ np.linalg.inv(A.T @ A) @ A.T @ v

print(f"additive (orthonormal) route: {np.round(proj_additive, 4)}")
print(f"hat-matrix route:             {np.round(proj_hat, 4)}")
print(f"agree to: {np.abs(proj_additive - proj_hat).max():.2e}")

## 4 · A semantic axis, built and used

A toy embedding table — hand-constructed so its structure is fully known (real studies use corpus-trained embeddings; the operations are identical). Dimensions are latent, but we built in a gender-like contrast and a status-like contrast. Build a gender axis from pairs, project occupation words, read the ranking.

In [ ]:
emb = {
    # pairs used to define axes
    "he":      np.array([ 2.1,  0.1,  0.3,  0.1]),
    "she":     np.array([-2.0,  0.2,  0.2,  0.0]),
    "man":     np.array([ 1.9, -0.1,  0.1,  0.2]),
    "woman":   np.array([-2.1,  0.0,  0.2,  0.1]),
    "king":    np.array([ 1.7,  1.8,  0.2,  0.1]),   # gender + royalty stowaway
    "queen":   np.array([-1.8,  1.9,  0.1,  0.2]),
    # occupation words to score
    "nurse":     np.array([-1.1,  0.2,  1.4,  0.3]),
    "engineer":  np.array([ 0.9,  0.1,  1.5,  0.2]),
    "teacher":   np.array([-0.3,  0.0,  1.3,  0.4]),
    "banker":    np.array([ 0.6,  0.9,  1.2,  0.1]),
}

def axis_from_pairs(pairs):
    v = np.mean([emb[a] - emb[b] for a, b in pairs], axis=0)
    return v / np.linalg.norm(v)          # normalise: the axis is a unit ruler

def score(word, axis):
    return emb[word] @ axis

gender = axis_from_pairs([("he", "she"), ("man", "woman")])
occupations = ["nurse", "engineer", "teacher", "banker"]
for w in sorted(occupations, key=lambda w: score(w, gender)):
    print(f"{w:10s} {score(w, gender):+.2f}")

## 5 · Now break it deliberately

Swap the defining pairs — use *king–queen* with its royalty stowaway — and watch the ranking move. Same words, same embeddings, different instrument.

In [ ]:
gender_alt = axis_from_pairs([("king", "queen"), ("man", "woman")])

print(f"{'word':10s} {'he/she axis':>12s} {'king/queen axis':>16s}")
for w in occupations:
    print(f"{w:10s} {score(w, gender):+12.2f} {score(w, gender_alt):+16.2f}")

print(f"\ncosine between the two 'gender' axes: {gender @ gender_alt:.3f}   <- not the same direction")

The *king–queen* pair drags a royalty component into the axis (look at `banker`, whose status loading now contaminates its "gender" score). The lesson is the module's: the axis is an **instrument**, its construction is a researcher choice, and results must be reported with sensitivity to that choice — many pairs, averaged, with the spread disclosed.

---

**Next:** M4 · Eigenvectors, SVD, and Low Rank — the module where matrices come apart, and low-rank adaptation becomes inevitable.